# Qwenx86-4B compact — in-weight x86 CPU on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/qwenx86-4b-compact-in-weight-cpu-ccbf/colab/Qwenx86-4B-compact-in-weight-cpu.ipynb)

[`Quazim0t0/Qwenx86-4B-compact`](https://huggingface.co/Quazim0t0/Qwenx86-4B-compact) is **Qwenx86-4B compact**: [`Qwen/Qwen3.5-4B`](https://huggingface.co/Qwen/Qwen3.5-4B) plus **13 compiled (not learned) “chip” layers** that implement a **64-bit x86 computer in the weights**. Qwen invokes the computer with `<tool_call>`; the chip layers execute the tool. Calculator, MicroPython, a small C compiler (`qcc`), a notes store, science lookup, and Python-docs lookup are **firmware** baked into `model/boot/*.safetensors`.

**Why this model.** This is a teaching demo of *in-weight computation*, not a faster CPU. Tool runs cost tens of thousands of **machine tokens** (card: calc `(1200-350)/5` → **170** in 56,667 tokens / ~41 s). That is slow compared with a real calculator. The point is that the arithmetic happened inside the transformer, checked against a reference machine.

This is **inference**, not training. It is distinct from [`qwen3_5_4b_decision_email_triage.ipynb`](qwen3_5_4b_decision_email_triage.ipynb) (Unsloth decision-model fine-tune of `unsloth/Qwen3.5-4B`). Do **not** load this checkpoint with plain `AutoModelForCausalLM.from_pretrained` — the runtime is the repo’s compact loader (`qwenx86.loader.load` / `python -m qwenx86.run`).

**Licence:** Apache-2.0 (modified `Qwen/Qwen3.5-4B`; `LICENSE-Qwen3.5`). `chipformer/` is MIT (Qwenputer / Divyansh Agrawal). Not gated.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The [card](https://huggingface.co/Quazim0t0/Qwenx86-4B-compact) says the compact checkpoint uses **~9 GB GPU memory, “so it fits on a 16 GB GPU.”** A free T4 is about **15 GB**. That should fit; this notebook prints allocated / free VRAM after load so you can confirm. T4 is Turing **sm_75**: **no native bfloat16**, **no FlashAttention-2**. Follow the official compact 16 GB loader — do not invent a second quant. The Hub also has [`Quazim0t0/Qwenx86-4B-GGUF`](https://huggingface.co/Quazim0t0/Qwenx86-4B-GGUF); this notebook does not use it.

**Download / RAM.** First run snapshot-downloads the Hub repo into `/content/Qwenx86-4B-compact` (a real folder, not the HF cache root): `model/compact.safetensors` (~**8.75 GB**) plus six `model/boot/*.safetensors` firmware states (~27 MB each), `machines/` (`tools.json` + `*.pkl`), and the `qwenx86/` + `chipformer/` runtime. Warn if free disk is tight (free Colab is usually tens of GB, but not infinite). Host RAM (~12.7 GB on free Colab) can get tight if a loader copies the whole shard to CPU first — this notebook uses the official compact loader (`safe_open` + per-tensor `.to("cuda")`).

**Speed.** Machine-token tool runs are a teaching demo, not a speedup over a host CPU. Keep the live cells short. Science is ~129 s / 402k tokens on the card; a long C compile is worse — those stay optional.

**Not in this notebook:** v2 host-driven memory, v3 `memory_*` chat, `python -m qwenx86.mixed`, the GGUF build, or the [debugger Space](https://huggingface.co/spaces/Quazim0t0/Qwenx86-Debugger) recorder.

**Links:** [model card](https://huggingface.co/Quazim0t0/Qwenx86-4B-compact) · [debugger Space](https://huggingface.co/spaces/Quazim0t0/Qwenx86-Debugger) · [Qwenputer](https://github.com/divagr18/qwenputer) · [tweet that surfaced it](https://x.com/TeksEdge/status/2118857211992854892)


## 1. Install card deps (keep Colab’s torch)

The card is `pip install -r requirements.txt` then `python -m qwenx86.run --tool calc ...` from the Hub repo. `requirements.txt` is `torch>=2.8`, `transformers==5.22.1`, `safetensors>=0.5`, `tokenizers>=0.23`, `numpy>=1.26`, `pillow>=10`, `huggingface_hub>=0.30`.

**Do not replace Colab’s CUDA torch** if it already works. `torch>=2.8` is the card floor — this cell prints the wheel and warns if it is older, but it will not `pip install torch`. Use **`%pip`**, not `python -I -m pip`. Isolated mode drops Colab site-packages and breaks the preinstalled torch.

The `qwenx86` package is **not on PyPI**. It comes from the Hub snapshot in the load cell.

Needs **Python 3.12** (card) and **CUDA**. Do **not** install `flash-attn`, `flash-linear-attention`, or `causal-conv1d`. `qwenx86.run.setup()` unwraps Hugging Face’s own pure-torch gated-delta / conv1d kernels.


In [ ]:
%pip install -q "transformers==5.22.1" "safetensors>=0.5" "tokenizers>=0.23" "numpy>=1.26" "pillow>=10" "huggingface_hub>=0.30"

import sys

print("python", sys.version.split()[0])
if sys.version_info < (3, 12):
    print("warning: the card asks for Python 3.12; this runtime is", sys.version.split()[0])

try:
    import torch

    print("torch (keeping Colab CUDA wheel)", torch.__version__, "| cuda", torch.version.cuda)
    # Card floor is torch>=2.8. Do not pip-install a different wheel over Colab's CUDA build.
    ver = torch.__version__.split("+")[0]
    major_minor = tuple(int(p) for p in ver.split(".")[:2])
    if major_minor < (2, 8):
        print("warning: card lists torch>=2.8; leaving Colab's wheel in place")
except ImportError:
    print("torch not imported yet — Colab should already have a CUDA wheel")

# Use %pip, not python -I -m pip (isolated mode drops Colab site-packages).
# Do NOT install flash-attn (T4 / sm_75 cannot run FlashAttention 2).
# Do NOT install flash-linear-attention or causal-conv1d.
print("install done — restart the runtime before the GPU-check cell")


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The pin to **5.22.1** is invisible until the process restarts.


## 2. Check the GPU

Print `torch.cuda.is_available()`, the GPU name, and free/allocated VRAM. A free T4 (~15 GB, `sm_75`) is the path this notebook is written for. The next cell stops if you are on CPU — the compact loader is CUDA (`qwenx86.run` needs a GPU; the “CPU” in the filename is the **in-weight x86 computer**, not a Colab CPU runtime).


In [ ]:
import gc
import shutil
import sys

import torch
import transformers
from packaging.version import Version

print("python", sys.version.split()[0])
print("cuda available:", torch.cuda.is_available())
assert Version(transformers.__version__.split("+")[0]) >= Version("5.22.1"), (
    f"transformers {transformers.__version__} is loaded; Qwenx86-4B compact needs "
    "transformers==5.22.1. Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2 and no native bfloat16 tensor cores.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

DEVICE = "cuda"
MODEL_ID = "Quazim0t0/Qwenx86-4B-compact"
major, minor = torch.cuda.get_device_capability()
try:
    NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    NATIVE_BF16 = major >= 8

free_b, total_b = torch.cuda.mem_get_info()
disk = shutil.disk_usage("/")
print("transformers", transformers.__version__, "| torch", torch.__version__)
print("device:", torch.cuda.get_device_name(0), f"sm_{major}{minor}")
print(f"VRAM free: {free_b / 1024**3:.2f} GB of {total_b / 1024**3:.1f} GB")
print(f"disk free: {disk.free / 1024**3:.1f} GB of {disk.total / 1024**3:.1f} GB")
print("native bf16:", NATIVE_BF16, "| FlashAttention-2: off")
print("card: compact checkpoint uses ~9 GB GPU, fits a 16 GB GPU; T4 is ~15 GB")
if disk.free < 12 * 1024**3:
    print("warning: compact.safetensors is ~8.75 GB plus boots — disk looks tight")
if major < 8:
    print("T4 path: official compact CUDA loader (no extra quant). bfloat16 tensor cores off.")


def print_vram(prefix=""):
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"{prefix}VRAM allocated: {allocated:.2f} GB | reserved: {reserved:.2f} GB | "
        f"peak: {peak:.2f} GB | free: {free_now / 1024**3:.2f}/{total_now / 1024**3:.2f} GB"
    )


print_vram("before load  ")


## 3. Snapshot the Hub repo and load with `qwenx86`

`snapshot_download("Quazim0t0/Qwenx86-4B-compact", local_dir="/content/Qwenx86-4B-compact")` pulls the compact weights, boot firmware, `qwenx86/`, `chipformer/`, and `machines/` (`tools.json` + `*.pkl`) into a **real folder**. Do not use the default HF cache path: `qwenx86` sets `MACHINES` from `Path(__file__).resolve().parent.parent / "machines"`, and cache blob symlinks resolve that to `models--…/machines/tools.json` (the cache root, not `snapshots/<rev>/`). This cell `chdir`s into `/content/Qwenx86-4B-compact`, puts that directory on `sys.path`, and asserts the loader files exist before import.

The card’s CLI is:

```bash
python -m qwenx86.run --tool calc    "What is (1200 - 350) / 5? Use the calc tool."
python -m qwenx86.run --tool python  "What is 2 to the power 100 exactly? Use the python tool."
```

There is no separate high-level `run_tool` on PyPI. This cell calls the same Python API the CLI uses (`qwenx86.run.setup`, compact `loader.load` via `RvChat`, `blockrun.generate`) so the **~9 GB model is loaded once** and VRAM can be printed. Subprocessing `python -m qwenx86.run` would reload the checkpoint every demo.

`loader.use_in_rvchat()` patches `Qwen3_5ForCausalLM.from_pretrained` to the compact 16 GB loader (`model/compact.safetensors`, nonzero rows/columns only — not an approximation). Card load time is ~25 s.


In [ ]:
import copy
import json
import os
import pickle
import sys
import time
from pathlib import Path

from huggingface_hub import snapshot_download

print_vram("before snapshot  ")
print("snapshot_download", MODEL_ID, "— ~9 GB compact + boots; this is not a second quant")
# Real folder, not the HF cache root (models--…/). qwenx86.MACHINES is
# Path(__file__).resolve().parent.parent / "machines"; cache blob
# symlinks resolve to models--…/machines/tools.json, which does not exist.
LOCAL_DIR = Path("/content/Qwenx86-4B-compact")
if not LOCAL_DIR.parent.is_dir():
    LOCAL_DIR = Path.cwd() / "Qwenx86-4B-compact"
SNAPSHOT = Path(
    snapshot_download(
        MODEL_ID,
        local_dir=str(LOCAL_DIR),
        # Runtime trees the compact loader needs. Include machines/
        # (tools.json + *.pkl) — do not skip it with a too-narrow glob.
        allow_patterns=[
            "qwenx86/**",
            "chipformer/**",
            "machines/*",
            "machines/**",
            "model/*",
            "model/**",
            "config.json",
            "requirements.txt",
        ],
    )
)
os.chdir(SNAPSHOT)
os.environ["QWENX86_MODEL"] = str(SNAPSHOT / "model")
if str(SNAPSHOT) not in sys.path:
    sys.path.insert(0, str(SNAPSHOT))

needed = [
    SNAPSHOT / "machines" / "tools.json",
    SNAPSHOT / "machines" / "calc.pkl",
    SNAPSHOT / "machines" / "python.pkl",
    SNAPSHOT / "machines" / "c.pkl",
    SNAPSHOT / "machines" / "notes.pkl",
    SNAPSHOT / "machines" / "science.pkl",
    SNAPSHOT / "machines" / "pydoc.pkl",
    SNAPSHOT / "model" / "compact.safetensors",
    SNAPSHOT / "model" / "config.json",
    SNAPSHOT / "model" / "tokenizer.json",
    SNAPSHOT / "model" / "boot" / "calc.safetensors",
    SNAPSHOT / "model" / "boot" / "python.safetensors",
    SNAPSHOT / "qwenx86" / "__init__.py",
    SNAPSHOT / "qwenx86" / "loader.py",
    SNAPSHOT / "chipformer" / "__init__.py",
]
missing = [str(p) for p in needed if not p.is_file()]
assert not missing, f"loader files missing under {SNAPSHOT}: {missing}"

import qwenx86  # noqa: E402  (paths, specials, 64 KB RAM map)
from qwenx86 import MACHINES, MODEL, loader  # noqa: E402
from qwenx86 import run as qxrun  # noqa: E402

assert Path(MACHINES).resolve() == (SNAPSHOT / "machines").resolve(), (
    f"qwenx86.MACHINES resolved to {MACHINES}, expected {SNAPSHOT / 'machines'}"
)
assert Path(MODEL).resolve() == (SNAPSHOT / "model").resolve(), (
    f"qwenx86.MODEL resolved to {MODEL}, expected {SNAPSHOT / 'model'}"
)

compact = MODEL / "compact.safetensors"
boots = sorted((MODEL / "boot").glob("*.safetensors"))
print("snapshot:", SNAPSHOT)
print("cwd:", Path.cwd())
print(f"compact.safetensors: {compact.stat().st_size / 1024**3:.2f} GB (card ~8.75 GB)")
print(f"boot firmware: {len(boots)} files, {sum(p.stat().st_size for p in boots) / 1024**2:.0f} MB")
print("loader: official compact 16 GB path (qwenx86.loader.load), device=cuda")
print("machines/tools.json:", MACHINES / "tools.json")

BR, RC, core = qxrun.setup()
TOOLS = json.loads((MACHINES / "tools.json").read_text(encoding="utf-8"))
NOTES_DIR = SNAPSHOT / "qwenx86_notes"
T = BR.Text(MODEL / "tokenizer.json")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print_vram("before RvChat  ")
t0 = time.time()
chat = RC.RvChat(MODEL, fast=True, executor=False)
chat.eos = qxrun._QwenEos(chat)
print(f"[loaded in {time.time() - t0:.0f} s]  (card: ~25 s)")
print("model dtype (first param):", next(chat.model.parameters()).dtype)
print("device:", next(chat.model.parameters()).device)
print_vram("after load  ")
print("card assumption: ~9 GB GPU memory on a 16 GB GPU; T4 is ~15 GB — compare allocated above")


def run_tool(tool_name: str, prompt: str, max_new: int = 400):
    """Same path as `python -m qwenx86.run --tool …`, reusing the loaded RvChat."""
    t = TOOLS[tool_name]
    RC.core = qxrun._Core(core, t["x86"])
    machine = pickle.loads((MACHINES / t["machine"]).read_bytes())
    shard = MODEL / "boot" / t["boot"]
    state = NOTES_DIR / "state.safetensors"
    if tool_name == "notes" and state.exists():
        machine = pickle.loads((NOTES_DIR / "machine.pkl").read_bytes())
        shard = state
    RC.hf.inject = lambda c, cache, _shard=shard: loader.boot(cache, _shard)
    ids = chat.prompt_ids([{"role": "user", "content": prompt}], tools=t["schema"])
    lock = BR.Session(T, copy.deepcopy(machine))
    lock.history(ids)
    res = chat.generate(
        ids,
        copy.deepcopy(machine),
        max_new=max_new,
        max_chip=50_000_000,
        lock=lock,
    )
    print(chat.text_of(res["tokens"]))
    ok = res["lock_error"] is None
    print(
        f"\n[{res['chip_tokens']} machine tokens in {res.get('blocks', 0)} verified blocks, "
        f"{res['text_tokens']} text tokens, {res['seconds']:.0f} s; "
        f"{'every machine token = the reference' if ok else 'LOCK ERROR: ' + json.dumps(res['lock_error'], default=str)}]"
    )
    if tool_name == "notes" and ok:
        NOTES_DIR.mkdir(exist_ok=True)
        loader.save_state(chat, state)
        (NOTES_DIR / "machine.pkl").write_bytes(pickle.dumps(lock.m))
    print_vram(f"after {tool_name}  ")
    return res


## 4. Demo — calc (card live-chat)

Card live chat: *“What is (1200 - 350) / 5? Use the calc tool.”*

| | |
|---|---|
| Expected answer | **170** |
| Machine tokens | 56,667 |
| Wall time (card, 16 GB) | ~41 s |
| Lock | 0 errors |

This is the required path. Tens of seconds is expected — the chip layers are stepping an in-weight calculator, not calling `eval()` on the host.


In [ ]:
CALC_PROMPT = "What is (1200 - 350) / 5? Use the calc tool."
print("prompt:", CALC_PROMPT)
print("expected: 170  (card: 56,667 machine tokens, ~41 s)")
run_tool("calc", CALC_PROMPT)


## 5. Demo — python `2**100` (short)

Card live chat: *“What is 2 to the power 100 exactly? Use the python tool.”*

| | |
|---|---|
| Expected answer | **1267650600228229401496703205376** |
| Machine tokens | 39,526 |
| Wall time (card) | ~46.8 s |

Still a teaching demo. MicroPython here is Qwenputer’s RISC-V firmware (the python tool is the one v1 tool that is not on the x86 path).


In [ ]:
PY_PROMPT = "What is 2 to the power 100 exactly? Use the python tool."
print("prompt:", PY_PROMPT)
print("expected: 1267650600228229401496703205376  (card: 39,526 machine tokens, ~47 s)")
run_tool("python", PY_PROMPT)


## 6. Optional extra tools (commented — do not Run all)

Leave `RUN_OPTIONAL = None` so *Runtime > Run all* stays on calc + python. Set it to one name if you want to wait.

| Tool | Card live-chat prompt | Card cost |
|---|---|---|
| `notes` | “Save a note called bike. The bike lock code is 7193.” | 117,746 tokens / ~82 s (put); a follow-up search is another ~75 k |
| `pydoc` | “What is the signature of math.gcd? Use the python_doc tool.” | 55,684 tokens / ~44 s |
| `science` | “What is the Avogadro constant? Use the science tool.” | **402,834 tokens / ~129 s** — skip unless you mean to wait |
| `c` | “Write a C program that prints 2 to the power 40. Run it with the c tool.” | long compile+run (card’s shorter `6*7` C chat was 169 k tokens / ~71 s) |

Same CLI as the card (`python -m qwenx86.run --tool …`) if you would rather subprocess from the snapshot directory — that **reloads** the 9 GB checkpoint, so prefer `run_tool` above.


In [ ]:
# Set to one of: "notes", "pydoc", "science", "c". Leave None so Run all stays short.
RUN_OPTIONAL = None

OPTIONAL_PROMPTS = {
    "notes": "Save a note called bike. The bike lock code is 7193.",
    "pydoc": "What is the signature of math.gcd? Use the python_doc tool.",
    "science": "What is the Avogadro constant? Use the science tool.",
    "c": "Write a C program that prints 2 to the power 40. Run it with the c tool.",
}

if RUN_OPTIONAL is None:
    print("optional extra tools skipped (RUN_OPTIONAL is None)")
else:
    prompt = OPTIONAL_PROMPTS[RUN_OPTIONAL]
    print("optional tool:", RUN_OPTIONAL)
    print("prompt:", prompt)
    run_tool(RUN_OPTIONAL, prompt)


## 7. Caveats and cleanup

**What this notebook is for.** A free-T4 inference demo of Qwenx86-4B compact: snapshot the Hub repo, load with the official `qwenx86` compact loader, print VRAM (~9 GB card assumption on a 16 GB GPU; T4 ~15 GB), and run the card’s short calc chat (expected **170**) plus python `2**100`.

**What it is not.** A host-CPU runtime (the filename’s “cpu” is the **in-weight x86 computer**). A speedup over a real calculator or CPython. The [Qwen3.5-4B email-triage trainer](qwen3_5_4b_decision_email_triage.ipynb). A GGUF / llama.cpp / vLLM path. v2/v3 memory tools.

**Loader.** Snapshot into `/content/Qwenx86-4B-compact` (not the HF cache root), then `qwenx86.loader.load` / `python -m qwenx86.run`. Do not swap in `AutoModelForCausalLM`. Do not invent another quant. `transformers==5.22.1`, CUDA, card `torch>=2.8` (Colab’s wheel is kept).

**T4 / kernels.** Official compact CUDA loader. No FlashAttention-2, no `flash-linear-attention` / `causal-conv1d` install. `setup()` unwraps HF’s pure-torch kernels. T4 has no native bfloat16 tensor cores.

**VRAM / disk / RAM.** Compact weights ~8.75 GB plus ~160 MB of boot firmware. Card: **~9 GB GPU**, fits 16 GB; T4 is ~15 GB. Compare the “after load” VRAM print. First download is ~9 GB — check disk free. Host RAM can get tight if a loader materialises the whole shard on CPU; this notebook follows the official compact 16 GB loader.

**Speed.** Machine-token tool runs take tens of seconds (calc ~41 s, python ~47 s on the card). That is the demo. Science (~129 s) and long C compiles are optional.

**Licence.** Apache-2.0 for the modified Qwen3.5-4B weights. `chipformer/` is MIT.

**Links again:** [model card](https://huggingface.co/Quazim0t0/Qwenx86-4B-compact) · [debugger Space](https://huggingface.co/spaces/Quazim0t0/Qwenx86-Debugger) · [tweet source](https://x.com/TeksEdge/status/2118857211992854892)

Free the GPU before you open another notebook:


In [ ]:
print_vram("before cleanup  ")
for name in ("chat", "res", "lock", "machine"):
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup  ")
print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print("Qwenx86-4B compact released. Re-run the load cell if you want another tool call.")
